# 05 — Partitioning, Shuffling, and Performance Tuning

Partitions, shuffle costs, caching, the Catalyst/Tungsten pipeline, Adaptive Query Execution, and the "my job is slow, how do you debug it" question that closes out most Spark interviews.

> **Setup note:** these notebooks are written but **not executed** — PySpark is not
> installed in this environment. To run them locally:
>
> ```bash
> python -m venv .venv && source .venv/bin/activate
> pip install pyspark==3.5.1
> # Java 11/17 must be on PATH (java -version)
> jupyter notebook
> ```
>
> Everything below is correct, runnable PySpark — read it as a reference and run
> cell-by-cell once your environment is set up.

## 1. What is a partition?

A **partition** is a chunk of a DataFrame/RDD that lives on one executor and is processed by one task. Partition count controls parallelism: too few partitions under-utilizes the cluster (idle cores); too many creates excessive scheduling overhead and tiny tasks.

- Read-time partition count is usually driven by input file count/size/splittability (e.g. one partition per ~128 MB HDFS/Parquet block, or one per input file for non-splittable formats).
- After a **shuffle** (`groupBy`, `join`, `orderBy`, `repartition`), partition count is controlled by `spark.sql.shuffle.partitions` (default **200** — frequently wrong for small local jobs, which is why we set it to 8 in notebook 01).

In [ ]:
df = spark.range(0, 1_000_000, numPartitions=6)
print("partitions:", df.rdd.getNumPartitions())

# repartition: full shuffle, can increase or decrease partitions, produces
# evenly-sized partitions (hash or round-robin)
df_16 = df.repartition(16)
print("after repartition(16):", df_16.rdd.getNumPartitions())

# coalesce: no full shuffle — merges adjacent partitions, can only DECREASE
# the count, cheaper than repartition but can leave partitions uneven
df_2 = df_16.coalesce(2)
print("after coalesce(2):", df_2.rdd.getNumPartitions())

# repartition by column: co-locates rows with the same key — do this before
# a groupBy/join on that key if you're going to reuse the DataFrame multiple times
df_by_key = df.repartition(8, (col("id") % 8))

**`repartition` vs `coalesce` — a near-guaranteed interview question:**

| | `repartition(n)` | `coalesce(n)` |
|---|---|---|
| Shuffle | full shuffle (expensive) | no shuffle — merges existing partitions |
| Can increase partitions? | yes | no (only decreases) |
| Partition balance | even | can be uneven (merges neighbors) |
| Typical use | before a wide op, or to fix skew | reducing output file count before a write |

## 2. Shuffles — the most expensive thing in Spark

A shuffle happens whenever data must be **regrouped across partitions** — `groupBy`, non-broadcast `join`, `distinct`, `orderBy`, `repartition`. It means: write intermediate data to disk on every executor (shuffle write), then every reducer task pulls its share over the network (shuffle read). This is disk + network + serialization cost, and it's the #1 place Spark jobs get slow.

**Minimizing shuffles:**
- Filter and select columns **before** a join/groupBy, not after (though Catalyst pushes filters down automatically in most cases — verify with `.explain()`).
- Use broadcast joins to skip shuffling the large side (see notebook 03).
- Reuse a shuffle: if you `repartition("key")` once and do multiple operations on that key, Spark can sometimes reuse the shuffle output — check `.explain()` for `Exchange` nodes to see if a shuffle is repeated.

## 3. Caching / persisting

`cache()` (shorthand for `persist(StorageLevel.MEMORY_AND_DISK)`) stores a DataFrame's computed partitions so a **re-used** DataFrame doesn't get recomputed from scratch each time an action touches it. Only worth it when the same DataFrame is used in **multiple actions/branches** — caching something used exactly once just adds overhead.

Storage levels: `MEMORY_ONLY` (fastest, lost if it doesn't fit), `MEMORY_AND_DISK` (spills to disk if needed — the sensible default), `DISK_ONLY`, and `_SER` variants (serialized — less memory, more CPU to deserialize). Always `unpersist()` when done to free executor memory.

In [ ]:
from pyspark import StorageLevel

expensive = sales.groupBy("rep").agg(spark_sum("amount").alias("total")).persist(StorageLevel.MEMORY_AND_DISK)

expensive.count()                 # first action materializes + caches it
top = expensive.orderBy(col("total").desc()).first()   # reuses the cached result
bottom = expensive.orderBy("total").first()             # reuses it again

expensive.unpersist()

## 4. Catalyst optimizer and Tungsten — what `.explain()` is showing you

Every DataFrame query goes through four stages, all visible via `.explain(mode="formatted")` or `.explain(True)`:

1. **Parsed Logical Plan** — direct translation of your code/SQL, unresolved (column/table names not yet checked).
2. **Analyzed Logical Plan** — column and table references resolved against the catalog/schema.
3. **Optimized Logical Plan** — Catalyst's rule-based optimizations: predicate pushdown, column pruning, constant folding, join reordering.
4. **Physical Plan** — the actual execution strategy (which join algorithm, how many partitions, where `Exchange` = shuffle boundaries sit). **Tungsten** then handles the low-level execution: off-heap binary memory layout, whole-stage code generation (fusing multiple operators into a single compiled JVM function to avoid per-row virtual-call overhead).

In [ ]:
plan = employees.join(broadcast(departments), "dept_id").filter(col("dept_id") == 10)
plan.explain(mode="formatted")
# Look for: "BroadcastHashJoin", "PushedFilters", and whether the filter
# was pushed down before or after the join in the optimized plan.

## 5. Adaptive Query Execution (AQE) — Spark 3.x, on by default

AQE re-optimizes the plan **mid-query** using *actual* runtime statistics from completed shuffle stages, instead of relying only on pre-query estimates. Three main features:

1. **Coalescing shuffle partitions** — merges small post-shuffle partitions automatically, so you don't have to hand-tune `spark.sql.shuffle.partitions` for every job size.
2. **Skew join optimization** — detects a partition that's disproportionately large (from real shuffle stats) and splits it into smaller sub-partitions, joining each piece separately.
3. **Dynamic join strategy switching** — if a table turns out to be smaller than expected *after* filters are applied (not knowable from static stats), AQE can switch a sort-merge join to a broadcast join at runtime.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")               # default: true (Spark 3.x)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")     # default: true
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")  # default: true

## 6. The small-files problem

Writing with too many output partitions produces thousands of tiny files — expensive to list/open later (especially on object stores like S3, where listing is a network call per prefix) and wastes storage due to block-size overhead. Fix by `coalesce(n)` (or `spark.sql.adaptive.coalescePartitions` for AQE-aware shuffle writes) immediately before `.write`, or by setting a target file size when the engine supports it (e.g. Delta Lake's `optimizeWrite`).

In [ ]:
(
    sales
    .groupBy("region")
    .agg(spark_sum("amount").alias("total"))
    .coalesce(1)             # one output file per write — fine for small results;
                              # for large writes, pick a partition count that targets ~128MB/file
    # .write.mode("overwrite").parquet("/tmp/region_totals")
)

## 7. Interview Q&A: "a job is slow, how do you debug it?"

1. **Open the Spark UI** (Stages tab) — find the stage taking longest.
2. **Check task-level skew**: sort tasks by duration within that stage — one or two stragglers vs. all-uniform tells you skew vs. genuinely-large data.
3. **Check for spill** ("Shuffle spill (memory)" / "(disk)" metrics) — means a partition didn't fit in executor memory and had to serialize to disk mid-task; fix by increasing partition count (smaller partitions) or executor memory.
4. **Check the DAG for repeated/unnecessary shuffles** — look for extra `Exchange` nodes in `.explain()` that a broadcast join or a smarter repartition could remove.
5. **Check input partition count/size** — too many tiny partitions (task scheduling overhead) or too few huge ones (poor parallelism, spill risk).

## Summary

- `repartition` = full shuffle, can grow partitions, even sizes; `coalesce` = no shuffle, can only shrink, possibly uneven.
- Shuffles are the most expensive operation — minimize them, broadcast when possible.
- Cache only DataFrames reused across multiple actions; always unpersist.
- `.explain()` shows Catalyst's four plan stages; AQE re-optimizes at runtime using real shuffle statistics.
- Next: `06_structured_streaming_and_interview_problems.ipynb`.